# Convolutional layers: PyTorch and from scratch

The MLP in `neural_networks.ipynb` treats an 8×8 digit as **64 unrelated numbers**. This notebook
builds the layer that knows those numbers are a *picture*.

Following the `models.md` plan — `... Softmax on MNIST → MLP → Conv + Pool from scratch → LeNet ...` —
this covers the **Conv + Pool** rung:

1. **Why convolution** — what a dense layer gets wrong about images, and the three ideas that fix it.
2. **PyTorch baseline** — `nn.Conv2d` on the same Optical Digits dataset, plus two experiments that
   *demonstrate* the difference rather than asserting it.
3. **NumPy from scratch** — naive loops first (readable), then `im2col` (fast), then the backward
   pass, each verified against PyTorch to ~15 decimal places.
4. **Gradient check** and a full training run with hand-computed metrics.

Same dataset, same evaluation pattern, and the same "prove it rather than claim it" approach as the
neural networks notebook.

## Why convolution? What a dense layer gets wrong about images

### The problem, in one sentence

A dense layer has **no idea that pixel 9 sits directly below pixel 1**.

Look at the MLP's first weight matrix: `W1` is `(64, 128)`, so each hidden neuron holds 64 weights,
one per pixel, and every pixel is just "input number *i*". Nothing in that layer encodes which
pixels are neighbours. If you scrambled the 64 pixels — the *same* scramble applied to every image,
so the data stays perfectly consistent — the MLP would train just as well. The cell below runs
exactly that experiment.

That is a real cost. The network has to *learn from data* that nearby pixels are related, spending
training examples to rediscover something we already knew for free. And what it learns in one corner
of the image transfers nowhere else: a stroke detector learned at the top-left is useless at the
bottom-right, because those are different weights entirely.

### Three ideas that fix it

**1. Local receptive field.** Instead of connecting a neuron to all 64 pixels, connect it to a small
patch — say 3×3. Nearby pixels are what actually form edges and strokes; a pixel in the opposite
corner is almost never directly relevant. The neuron looks at a *window*, not the whole image.

**2. Parameter sharing.** Take that one 3×3 window of weights and slide it across every position in
the image. Every location is examined by the **same** 9 weights. Those 9 weights are called a
**filter** or **kernel**. This is the big idea, and it buys two things at once:

- *Far fewer parameters.* One filter is 9 weights instead of 64 per neuron. Weights no longer scale
  with image size — the same 3×3 filter works on an 8×8 digit or a 4000×3000 photo.
- *A detector that works everywhere.* Learn "this looks like a vertical edge" once, and you have it
  at all 64 positions for free, rather than having to relearn it at each one.

**3. Translation equivariance.** Shift the input one pixel right, and the output shifts one pixel
right too — the same features are found, just relocated. A dense layer has no such guarantee: shift
the image and every input lands on a different weight, so the answer can change completely. The
second experiment below shifts the test digits by one pixel and measures exactly this.

### Counting the difference

For our 8×8 digits, a dense layer producing 8 "feature" outputs per pixel position versus one conv
layer with 8 filters of 3×3:

| | weights | scales with image size? | detector reusable across positions? |
| --- | --- | --- | --- |
| Dense (64 → 512) | 32,768 | yes, quadratically | no |
| Conv (8 filters, 3×3) | **72** | **no** | **yes** |

72 weights, and each of the 8 filters is applied at all 64 positions — so the layer computes 512
outputs from 72 numbers. That ratio is the entire reason convolution is used on images, audio, and
anything else with a grid structure where *what* a pattern is matters more than *where* it is.

### The honest caveat, up front

Our digits are 8×8. Convolution's advantages grow with image size and with how much objects move
around — on 8×8 centred digits there is not much room for either. Expect the CNN here to roughly
*match* the MLP on accuracy while using far fewer parameters, and to win clearly only on the
shifted-image test. The point of this notebook is the mechanism, not a leaderboard score.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, log_loss)

torch.manual_seed(42)
np.random.seed(42)

# ── Data: the same 8x8 Optical Digits used in neural_networks.ipynb ──
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# A conv layer needs the image SHAPE back: (N, channels, height, width).
# The MLP flattened each digit to a length-64 vector and threw the geometry away;
# this is the step that gives it back.
#
# Pixel values run 0..16, so dividing by 16 puts them in [0, 1] -- the ordinary
# image convention. (StandardScaler would also work, but per-pixel standardization
# on images is unusual: it rescales each pixel position independently, which is
# exactly the position-specific thinking convolution is trying to avoid.)
def to_images(A):
    return torch.tensor(A / 16.0, dtype=torch.float32).reshape(-1, 1, 8, 8)

Xtr_img, Xte_img = to_images(X_train), to_images(X_test)
ytr_t = torch.tensor(y_train, dtype=torch.long)
yte_t = torch.tensor(y_test, dtype=torch.long)
print('train images:', tuple(Xtr_img.shape), '  test images:', tuple(Xte_img.shape))


class SmallCNN(nn.Module):
    """conv(1 -> 8 channels, 3x3, pad 1) -> ReLU -> maxpool 2x2 -> linear(128 -> 10)

    Deliberately tiny. One convolution is enough to show what convolution does,
    and it keeps the from-scratch NumPy version below readable.
    """
    def __init__(self):
        super().__init__()
        # 8 filters, each covering 1 input channel over a 3x3 window.
        # padding=1 adds a one-pixel border of zeros so 8x8 stays 8x8 -- without it
        # the output would shrink to 6x6 and the border pixels would be under-used.
        self.conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
        # After 2x2 pooling: 8 channels of 4x4 = 128 numbers -> 10 class scores.
        self.fc = nn.Linear(8 * 4 * 4, 10)

    def forward(self, x):
        x = F.relu(self.conv(x))    # (N, 1, 8, 8) -> (N, 8, 8, 8)
        x = F.max_pool2d(x, 2)      # (N, 8, 8, 8) -> (N, 8, 4, 4)
        x = x.flatten(1)            # (N, 8, 4, 4) -> (N, 128)
        return self.fc(x)           # (N, 128)     -> (N, 10)  raw logits


model = SmallCNN()
n_conv = sum(p.numel() for p in model.conv.parameters())
n_fc = sum(p.numel() for p in model.fc.parameters())
print(f'conv params: {n_conv:5d}   (8 filters x 1 channel x 3x3, + 8 biases)')
print(f'fc   params: {n_fc:5d}   (128 x 10, + 10 biases)')
print(f'total      : {n_conv + n_fc:5d}   vs 9,610 for the MLP in neural_networks.ipynb')

# CrossEntropyLoss = log-softmax + negative log likelihood, so the model returns
# raw logits and the softmax happens inside the loss (numerically safer).
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

batch_size, n_epochs = 64, 30
loss_curve = []
for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(len(Xtr_img))
    running = 0.0
    for s in range(0, len(perm), batch_size):
        idx = perm[s:s + batch_size]
        opt.zero_grad()
        loss = loss_fn(model(Xtr_img[idx]), ytr_t[idx])
        loss.backward()
        opt.step()
        running += loss.item() * len(idx)
    loss_curve.append(running / len(perm))
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f'Epoch {epoch:3d}: loss = {loss_curve[-1]:.4f}')

# ── Evaluate with the same metric set as the MLP notebook ────────────
model.eval()
with torch.no_grad():
    proba = F.softmax(model(Xte_img), dim=1).numpy()
y_pred = proba.argmax(axis=1)

print(f'\nAccuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Log loss: {log_loss(y_test, proba):.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=digits.target_names).plot(cmap='Blues')
plt.title('CNN (PyTorch): confusion matrix')
plt.show()

In [ ]:
# What did the 8 filters actually learn, and what do they respond to?
#
# Top row : the 8 learned 3x3 kernels. Each is 9 numbers. Red = positive weight,
#           blue = negative. A filter fires strongly where the image looks like
#           its red region and not like its blue region -- so a kernel with red
#           above blue is an edge detector for horizontal edges, and so on.
# Bottom  : the 8 feature maps for one test digit -- the output of sliding each
#           kernel over the image. Bright = that filter matched at that position.

W = model.conv.weight.detach().numpy()          # (8, 1, 3, 3)
sample = Xte_img[0:1]                            # one digit, shape (1, 1, 8, 8)
with torch.no_grad():
    feats = F.relu(model.conv(sample))[0].numpy()   # (8, 8, 8)

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
vmax = np.abs(W).max()
for k in range(8):
    axes[0, k].imshow(W[k, 0], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0, k].set_title(f'filter {k}', fontsize=9)
    axes[1, k].imshow(feats[k], cmap='viridis')
    axes[1, k].set_title(f'map {k}', fontsize=9)
    axes[2, k].axis('off')
for ax in axes[:2].ravel():
    ax.set_xticks([]); ax.set_yticks([])

# The input digit, for reference, centred under the maps.
axes[2, 3].imshow(sample[0, 0], cmap='gray')
axes[2, 3].set_title(f'input (true = {y_test[0]})', fontsize=9)
axes[2, 3].set_xticks([]); axes[2, 3].set_yticks([]); axes[2, 3].axis('on')

axes[0, 0].set_ylabel('kernels', fontsize=10)
axes[1, 0].set_ylabel('feature maps', fontsize=10)
plt.suptitle('Learned 3x3 kernels (top) and the feature maps they produce (middle)', fontsize=13)
plt.tight_layout()
plt.show()

print('Each kernel is just 9 numbers, reused at all 64 positions of the image.')
print(f'All 8 kernels together: {W.size} weights, producing {feats.size} output values.')

In [ ]:
# Two experiments that test the claims in the "Why convolution" section above.
#
# Everything is averaged over 3 seeds. With only 360 test images, one run wanders
# by about a point of accuracy on initialization noise alone -- which is the same
# size as the effect being measured. A single run here would be unreadable.

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEEDS = (0, 1, 2)
EPOCHS = 25


def make_shallow_cnn():
    """The SmallCNN architecture: one conv, then a 128 -> 10 fully-connected head."""
    return nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(8 * 4 * 4, 10))


def make_gap_cnn():
    """Two convs, then GLOBAL AVERAGE POOLING instead of a flatten.

    Averaging over all positions leaves the classifier 32 numbers -- one per
    channel, with position thrown away. So this head cannot learn "the value at
    index 37 matters"; it can only use what the conv layers actually found.
    That makes it a far sharper test of whether locality is being used at all.
    """
    return nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10))


def train_torch(make, Xtr_flat, ytr, seed):
    torch.manual_seed(seed)
    m = make()
    o = torch.optim.Adam(m.parameters(), lr=0.01)
    Xi, yt = to_images(Xtr_flat), torch.tensor(ytr, dtype=torch.long)
    for _ in range(EPOCHS):
        perm = torch.randperm(len(Xi))
        for s in range(0, len(perm), 64):
            i = perm[s:s + 64]
            o.zero_grad()
            nn.CrossEntropyLoss()(m(Xi[i]), yt[i]).backward()
            o.step()
    return m.eval()


def torch_acc(m, X_flat, y_true):
    with torch.no_grad():
        return accuracy_score(y_true, m(to_images(X_flat)).argmax(1).numpy())


def train_sk_mlp(Xtr_flat, ytr, seed):
    sc = StandardScaler().fit(Xtr_flat)
    clf = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500,
                        random_state=seed).fit(sc.transform(Xtr_flat), ytr)
    return sc, clf


def sk_acc(bundle, X_flat, y_true):
    sc, clf = bundle
    return accuracy_score(y_true, clf.predict(sc.transform(X_flat)))


# ── Experiment 1: shuffle the pixels ─────────────────────────────────
# ONE fixed permutation, applied to every image. No information is destroyed --
# it is a relabelling of the inputs -- but pixels that were neighbours no longer are.
rng_px = np.random.RandomState(0)
perm_px = rng_px.permutation(64)
Xtr_shuf, Xte_shuf = X_train[:, perm_px], X_test[:, perm_px]

# Train each model on normal data and on shuffled data (kept for experiment 2).
mlp_normal = [train_sk_mlp(X_train, y_train, s) for s in SEEDS]
mlp_shuf = [train_sk_mlp(Xtr_shuf, y_train, s) for s in SEEDS]
cnn_normal = [train_torch(make_shallow_cnn, X_train, y_train, s) for s in SEEDS]
cnn_shuf = [train_torch(make_shallow_cnn, Xtr_shuf, y_train, s) for s in SEEDS]
gap_normal = [train_torch(make_gap_cnn, X_train, y_train, s) for s in SEEDS]
gap_shuf = [train_torch(make_gap_cnn, Xtr_shuf, y_train, s) for s in SEEDS]

mean = lambda v: float(np.mean(v))
rows = [
    ('MLP (128 hidden)', mean([sk_acc(m, X_test, y_test) for m in mlp_normal]),
                         mean([sk_acc(m, Xte_shuf, y_test) for m in mlp_shuf])),
    ('CNN + dense head', mean([torch_acc(m, X_test, y_test) for m in cnn_normal]),
                         mean([torch_acc(m, Xte_shuf, y_test) for m in cnn_shuf])),
    ('CNN + GAP head  ', mean([torch_acc(m, X_test, y_test) for m in gap_normal]),
                         mean([torch_acc(m, Xte_shuf, y_test) for m in gap_shuf])),
]

print('Experiment 1 -- pixels shuffled (one fixed shuffle, mean of 3 seeds)')
print(f'{"":18s}{"normal":>9}{"shuffled":>10}{"change":>9}')
for name, a, b in rows:
    print(f'{name:18s}{a:9.4f}{b:10.4f}{b - a:+9.4f}')
print("""
  The MLP is unmoved -- and it cannot be otherwise: permuting the inputs just
  permutes the rows of W1, giving a network that is identical up to a relabelling.
  It never used adjacency, so it has nothing to lose.

  The CNN with a dense head loses only a little. Its 3x3 windows now group
  unrelated pixels, so the conv layer is close to useless -- but the 1,290-weight
  fully-connected head is strong enough to classify from the scrambled features
  anyway. The prior is broken; the head papers over it.

  The CNN with a global-average-pool head has no such escape: position is averaged
  away, so it can only use what convolution found. That is where the damage shows
  up clearly, and it is the honest measure of how much locality was worth here.
""")


# ── Experiment 2: shift the test digits by one pixel ─────────────────
# Train on normal images, test on images rolled one pixel right. The digit is
# unchanged; only its position moved.
def shift_right(X_flat, by=1):
    return np.roll(X_flat.reshape(-1, 8, 8), by, axis=2).reshape(-1, 64)


Xte_shift = shift_right(X_test, 1)

print('Experiment 2 -- test digits shifted right 1 pixel (trained unshifted, mean of 3 seeds)')
print(f'{"":18s}{"normal":>9}{"shifted":>10}{"change":>9}')
for name, models, acc_fn in [('MLP (128 hidden)', mlp_normal, sk_acc),
                             ('CNN + dense head', cnn_normal, torch_acc),
                             ('CNN + GAP head  ', gap_normal, torch_acc)]:
    a = mean([acc_fn(m, X_test, y_test) for m in models])
    b = mean([acc_fn(m, Xte_shift, y_test) for m in models])
    print(f'{name:18s}{a:9.4f}{b:10.4f}{b - a:+9.4f}')
print("""
  Both models degrade -- one pixel out of eight is a big shift, and none of these
  were trained with data augmentation. But the MLP collapses much further: every
  pixel now lands on a different weight, so almost nothing it learned still applies.
  The CNN finds the same strokes one position over and keeps far more of its
  accuracy. That gap is translation equivariance being useful.""")

## Convolution: the math

### Notation

- $\mathbf{X} \in \mathbb{R}^{N \times C_{in} \times H \times W}$: a batch of $N$ images, each with
  $C_{in}$ channels (1 for greyscale, 3 for RGB) of height $H$ and width $W$.
- $\mathbf{W} \in \mathbb{R}^{C_{out} \times C_{in} \times K_H \times K_W}$: the filters. There are
  $C_{out}$ of them, each spanning all $C_{in}$ input channels over a $K_H \times K_W$ window.
- $\mathbf{b} \in \mathbb{R}^{C_{out}}$: one bias per filter.
- $S$: stride (how far the window jumps each step). $P$: padding (border of zeros added).

### Forward pass

Each output pixel is a **dot product between one filter and one patch of the input**:

$$Y_{n,\,c_{out},\,i,\,j} \;=\; b_{c_{out}} \;+\; \sum_{c=1}^{C_{in}} \sum_{u=1}^{K_H} \sum_{v=1}^{K_W}
X_{n,\,c,\;iS+u-P,\;jS+v-P} \cdot W_{c_{out},\,c,\,u,\,v}$$

In words: **line the filter up over a patch, multiply element by element, add it all up, add the
bias, write one number. Slide over by $S$ and repeat.** That is the whole operation — everything
else is bookkeeping about where the window lands.

Two details worth naming:

- The sum over $c$ means a filter reads **all input channels at once**. A 3×3 filter on an RGB image
  is $3 \times 3 \times 3 = 27$ weights, and produces a single number per position — channels are
  collapsed, not kept separate.
- $C_{out}$ different filters produce $C_{out}$ output channels, called **feature maps**. Each one
  answers "where in the image does *my* pattern appear?"

### Output shape

$$H_{out} = \left\lfloor \frac{H + 2P - K_H}{S} \right\rfloor + 1, \qquad
W_{out} = \left\lfloor \frac{W + 2P - K_W}{S} \right\rfloor + 1$$

The formula just counts how many times the window fits. Useful special case: with $S=1$ and
$P = \lfloor K/2 \rfloor$ for odd $K$ (so $P=1$ for $K=3$), the output is the **same size** as the
input. That is why `padding=1` appears with `kernel_size=3` almost everywhere. Without padding, each
layer shrinks the image and pixels near the border get visited by fewer windows than pixels in the
middle.

### Parameter count

$$\#\text{params} = C_{out} \times C_{in} \times K_H \times K_W \;+\; C_{out}$$

Note what is **absent**: $H$ and $W$. A conv layer's size does not depend on the size of the image
it processes. A dense layer's does, quadratically. This is the parameter-sharing win, written down.

### A note on the name

What is defined above, and what every deep-learning framework calls "convolution", is technically
**cross-correlation**. True mathematical convolution flips the kernel first:

$$(f * g)[i] = \sum_u f[u]\, g[i - u] \qquad \text{(flipped)} \qquad\text{vs}\qquad
(f \star g)[i] = \sum_u f[u]\, g[i + u] \qquad \text{(not flipped)}$$

It makes no practical difference: the kernel is *learned*, so if a flip were needed the network would
simply learn the flipped weights. The name stuck for historical reasons. The distinction does
resurface in the backward pass below, where a genuine flip appears.

### Max pooling

$$Y_{n,c,i,j} = \max_{0 \le u,v < p} X_{n,\,c,\;iS+u,\;jS+v}$$

Take a small window (usually 2×2, stride 2, non-overlapping) and keep only the largest value. This
halves height and width, so it cuts the data by 4×. Two reasons it is used:

- **A small shift stops mattering.** If the strongest response moves by one pixel but stays inside
  the same 2×2 window, the pooled output is unchanged. Convolution gives *equivariance* (features
  move with the input); pooling converts some of that into *invariance* (the answer stops moving).
- **It widens the view cheaply.** After pooling, a later 3×3 filter covers a 6×6 region of the
  original image without any extra weights.

Pooling has **no parameters** — nothing to learn, just a max.

### Backward pass

Given $\frac{\partial J}{\partial \mathbf{Y}}$ flowing back from above, three gradients are needed.

**Bias** — each filter's bias was added once per output position, so sum over everything except the
filter axis:

$$\frac{\partial J}{\partial b_{c_{out}}} = \sum_{n}\sum_{i}\sum_{j} \frac{\partial J}{\partial Y_{n,c_{out},i,j}}$$

**Filters** — a weight $W_{c_{out},c,u,v}$ was used at *every* output position, so its gradient
collects a contribution from every one of them:

$$\frac{\partial J}{\partial W_{c_{out},c,u,v}} = \sum_{n}\sum_{i}\sum_{j}
\frac{\partial J}{\partial Y_{n,c_{out},i,j}} \cdot X_{n,\,c,\;iS+u-P,\;jS+v-P}$$

This is the price of parameter sharing: one weight, many uses, so many terms added together. It is
itself a convolution — of the input with the output gradient.

**Input** — a single input pixel was read by several overlapping windows, so its gradient sums over
all output positions that touched it:

$$\frac{\partial J}{\partial X_{n,c,h,w}} = \sum_{c_{out}}\sum_{u}\sum_{v}
\frac{\partial J}{\partial Y_{n,\,c_{out},\;(h+P-u)/S,\;(w+P-v)/S}} \cdot W_{c_{out},c,u,v}$$

This one is a **full convolution of the output gradient with the kernel flipped in both spatial
directions** — the flip that the forward pass did not do shows up here. Intuition: forward, a pixel
scatters its influence into several outputs; backward, it gathers gradient from those same outputs,
and gathering is the mirror image of scattering.

**Max pooling backward** — the max is a router. Only the winning element affected the output, so it
receives the entire gradient and every other element in the window receives zero:

$$\frac{\partial J}{\partial X_{n,c,h,w}} = \begin{cases}
\frac{\partial J}{\partial Y_{n,c,i,j}} & \text{if } (h,w) \text{ was the argmax of its window} \\
0 & \text{otherwise}
\end{cases}$$

### `im2col`: how it is made fast

Writing the forward pass as six nested loops is correct and unbearably slow. The standard trick is
**`im2col`**: copy every patch the filter will visit into a column of a big matrix. Then the whole
convolution is a **single matrix multiply**:

$$\mathbf{Y} = \underbrace{\mathbf{W}_{\text{flat}}}_{C_{out} \,\times\, C_{in}K_HK_W}
\cdot \underbrace{\text{im2col}(\mathbf{X})}_{C_{in}K_HK_W \,\times\, H_{out}W_{out}} \;+\; \mathbf{b}$$

It trades memory for speed — overlapping patches are duplicated, so the column matrix is larger than
the image — and it wins anyway, because decades of optimization have gone into matrix multiplication.
Both versions are implemented below, and checked against each other.

### Gradient check

Same tool as in `neural_networks.ipynb` — compare each analytic gradient with a numerical one:

$$\frac{\partial J}{\partial w} \approx \frac{J(w+\epsilon) - J(w-\epsilon)}{2\epsilon}, \qquad
\text{relative error} = \frac{|g_{\text{analytic}} - g_{\text{numerical}}|}
{|g_{\text{analytic}}| + |g_{\text{numerical}}| + \epsilon_{\text{small}}} < 10^{-5}$$

Here there is a second, stronger check available: PyTorch's autograd computes the same gradients, so
the from-scratch results can be compared against it directly.

## Convolution from scratch (NumPy only)

Three steps, each verified before moving on:

1. **Naive forward** — six nested loops, written to match the formula line for line.
2. **`im2col` forward** — the same result as one matrix multiply, and much faster.
3. **Backward pass** — `dX`, `dW`, `db` for convolution, plus max pooling both ways.

PyTorch is used *only as a reference implementation* to check the NumPy against — never to compute
anything the from-scratch code needs.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# ── Naive convolution: the formula, written out as loops ─────────────
# This is deliberately the slow, obvious version. Every line maps to a piece of
#   Y[n, co, i, j] = b[co] + sum_c sum_u sum_v  X[n, c, i*S+u-P, j*S+v-P] * W[co, c, u, v]
# Read it once here, and the fast version below is easy to trust.
def conv2d_naive(x, W, b, stride=1, pad=0):
    """x: (N, C_in, H, W_)   W: (C_out, C_in, KH, KW)   b: (C_out,)"""
    N, C_in, H, W_ = x.shape
    C_out, _, KH, KW = W.shape

    # Padding = a border of zeros. Done once, up front, so the loops below never
    # have to worry about falling off the edge of the image.
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')

    # How many times does the window fit? -> the output-shape formula.
    H_out = (H + 2 * pad - KH) // stride + 1
    W_out = (W_ + 2 * pad - KW) // stride + 1

    out = np.zeros((N, C_out, H_out, W_out), dtype=x.dtype)
    for n in range(N):                          # each image
        for co in range(C_out):                 # each filter
            for i in range(H_out):              # each output row
                for j in range(W_out):          # each output column
                    h0, w0 = i * stride, j * stride
                    # One patch: all input channels, KH x KW spatial window.
                    patch = xp[n, :, h0:h0 + KH, w0:w0 + KW]     # (C_in, KH, KW)
                    # Multiply elementwise with the filter and add it all up.
                    # This single number is one pixel of one feature map.
                    out[n, co, i, j] = np.sum(patch * W[co]) + b[co]
    return out


# ── Check it against PyTorch ─────────────────────────────────────────
# float64 on both sides, so any disagreement is a real bug rather than float32 noise.
rng = np.random.RandomState(0)
x = rng.randn(2, 3, 7, 7)          # 2 images, 3 channels, 7x7  (multi-channel on purpose)
Wf = rng.randn(4, 3, 3, 3)         # 4 filters, 3 channels, 3x3
bf = rng.randn(4)

for stride, pad in [(1, 0), (1, 1), (2, 1), (2, 0)]:
    mine = conv2d_naive(x, Wf, bf, stride=stride, pad=pad)
    ref = F.conv2d(torch.tensor(x), torch.tensor(Wf), torch.tensor(bf),
                   stride=stride, padding=pad).numpy()
    err = np.abs(mine - ref).max()
    print(f'stride={stride} pad={pad}  out shape {mine.shape}  '
          f'max |diff| vs torch = {err:.2e}  {"OK" if err < 1e-12 else "MISMATCH"}')

# The output-shape formula, verified rather than asserted:
print()
for H, K, S, P in [(8, 3, 1, 1), (8, 3, 1, 0), (8, 3, 2, 1), (28, 5, 1, 0), (32, 3, 2, 1)]:
    predicted = (H + 2 * P - K) // S + 1
    actual = conv2d_naive(rng.randn(1, 1, H, H), rng.randn(1, 1, K, K),
                          np.zeros(1), stride=S, pad=P).shape[-1]
    print(f'H={H:3d} K={K} S={S} P={P} -> formula says {predicted:3d}, actual {actual:3d}  '
          f'{"OK" if predicted == actual else "MISMATCH"}')

In [ ]:
import time

# ── im2col: turn "slide a window" into "one matrix multiply" ─────────
# For every position the filter will visit, copy that patch out and stack the
# patches as columns. A (C_in, KH, KW) patch becomes a column of C_in*KH*KW numbers.
#
# The loops below run over the KERNEL (3x3 = 9 iterations), not over the image
# positions (which could be thousands) -- each iteration grabs one kernel offset
# from every window at once, using strided slicing.
def im2col(x, KH, KW, stride=1, pad=0):
    """(N, C, H, W) -> cols (N, C*KH*KW, H_out*W_out), plus the output shape."""
    N, C, H, W_ = x.shape
    H_out = (H + 2 * pad - KH) // stride + 1
    W_out = (W_ + 2 * pad - KW) // stride + 1
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)), mode='constant')

    cols = np.zeros((N, C, KH, KW, H_out, W_out), dtype=x.dtype)
    for u in range(KH):
        for v in range(KW):
            # Every window's (u, v) element, for all output positions at once.
            cols[:, :, u, v, :, :] = xp[:, :, u:u + stride * H_out:stride,
                                            v:v + stride * W_out:stride]
    return cols.reshape(N, C * KH * KW, H_out * W_out), (H_out, W_out)


def col2im(cols, x_shape, KH, KW, stride, pad, H_out, W_out):
    """The inverse of im2col, used by the backward pass.

    Not a true inverse: a pixel copied into several overlapping patches must have
    those contributions ADDED back together, which is exactly what the chain rule
    asks for when one value feeds several outputs.
    """
    N, C, H, W_ = x_shape
    xp = np.zeros((N, C, H + 2 * pad, W_ + 2 * pad), dtype=cols.dtype)
    cols = cols.reshape(N, C, KH, KW, H_out, W_out)
    for u in range(KH):
        for v in range(KW):
            xp[:, :, u:u + stride * H_out:stride,
                     v:v + stride * W_out:stride] += cols[:, :, u, v]
    return xp[:, :, pad:pad + H, pad:pad + W_] if pad > 0 else xp


def conv2d_fast(x, W, b, stride=1, pad=0):
    """Same maths as conv2d_naive, as one matmul. Returns the cache for backward."""
    N = x.shape[0]
    C_out, C_in, KH, KW = W.shape
    cols, (H_out, W_out) = im2col(x, KH, KW, stride, pad)     # (N, C*KH*KW, L)
    W_flat = W.reshape(C_out, -1)                             # (C_out, C*KH*KW)
    # For each image n: W_flat @ cols[n]  ->  (C_out, L). einsum does all n at once.
    out = np.einsum('oc,ncl->nol', W_flat, cols) + b[None, :, None]
    return out.reshape(N, C_out, H_out, W_out), (x.shape, cols, W, stride, pad, H_out, W_out)


# ── Same answer as the naive version? ────────────────────────────────
for stride, pad in [(1, 0), (1, 1), (2, 1)]:
    slow = conv2d_naive(x, Wf, bf, stride, pad)
    fast, _ = conv2d_fast(x, Wf, bf, stride, pad)
    ref = F.conv2d(torch.tensor(x), torch.tensor(Wf), torch.tensor(bf),
                   stride=stride, padding=pad).numpy()
    print(f'stride={stride} pad={pad}   naive vs im2col: {np.abs(slow - fast).max():.2e}   '
          f'im2col vs torch: {np.abs(fast - ref).max():.2e}')

# ── And how much faster? ─────────────────────────────────────────────
xb = rng.randn(64, 3, 32, 32)      # a realistic minibatch
Wb = rng.randn(16, 3, 3, 3)
bb = rng.randn(16)

t0 = time.perf_counter(); conv2d_naive(xb, Wb, bb, 1, 1); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); conv2d_fast(xb, Wb, bb, 1, 1);  t_fast = time.perf_counter() - t0
t0 = time.perf_counter()
F.conv2d(torch.tensor(xb), torch.tensor(Wb), torch.tensor(bb), padding=1)
t_torch = time.perf_counter() - t0

print(f'\n64 images, 3 channels, 32x32, 16 filters:')
print(f'  naive loops : {t_naive * 1000:8.1f} ms')
print(f'  im2col      : {t_fast * 1000:8.1f} ms   ({t_naive / t_fast:5.1f}x faster than naive)')
print(f'  torch       : {t_torch * 1000:8.1f} ms   ({t_naive / t_torch:5.1f}x faster than naive)')
print('\nSame arithmetic in all three -- only the memory layout differs.')

In [ ]:
# ── Convolution backward ─────────────────────────────────────────────
# Three gradients, matching the three formulas in the theory section.
# Working in im2col space keeps all three as matrix multiplies.
def conv2d_backward(dout, cache):
    """dout: (N, C_out, H_out, W_out)  ->  dx, dW, db"""
    x_shape, cols, W, stride, pad, H_out, W_out = cache
    N = dout.shape[0]
    C_out, C_in, KH, KW = W.shape
    dout_flat = dout.reshape(N, C_out, -1)              # (N, C_out, L)

    # db: the bias was added once per output position, so sum over everything
    #     except the filter axis.
    db = dout_flat.sum(axis=(0, 2))                     # (C_out,)

    # dW: each weight was used at EVERY output position, so its gradient collects
    #     a term from each -- gradient x the input patch it multiplied.
    #     This is the cost of parameter sharing, and the reason conv layers train
    #     on so few parameters: many gradient signals, one weight.
    dW_flat = np.einsum('nol,ncl->oc', dout_flat, cols)  # (C_out, C_in*KH*KW)
    dW = dW_flat.reshape(W.shape)

    # dx: scatter each output's gradient back onto the patch that produced it,
    #     then col2im ADDS the overlapping contributions. That addition is the
    #     "sum over all windows that saw this pixel" from the formula -- and it is
    #     what the flipped-kernel full convolution amounts to in practice.
    dcols = np.einsum('oc,nol->ncl', W.reshape(C_out, -1), dout_flat)
    dx = col2im(dcols, x_shape, KH, KW, stride, pad, H_out, W_out)
    return dx, dW, db


# ── Max pooling, forward and backward ────────────────────────────────
# Non-overlapping p x p windows (stride = p), the common case.
# The argmax INDEX is stored, because backward has to send the gradient back to
# exactly the element that won -- and to no other.
def maxpool_forward(x, p=2):
    N, C, H, W_ = x.shape
    assert H % p == 0 and W_ % p == 0, 'pool size must divide the spatial dims'
    # Regroup so each p*p window sits in the last axis, then take the max there.
    xr = (x.reshape(N, C, H // p, p, W_ // p, p)
            .transpose(0, 1, 2, 4, 3, 5)
            .reshape(N, C, H // p, W_ // p, p * p))
    idx = xr.argmax(axis=-1)                                       # who won
    out = np.take_along_axis(xr, idx[..., None], axis=-1).squeeze(-1)
    return out, (idx, x.shape, p)


def maxpool_backward(dout, cache):
    """The max is a router: the winner gets all the gradient, everyone else zero."""
    idx, shape, p = cache
    N, C, H, W_ = shape
    dxr = np.zeros((N, C, H // p, W_ // p, p * p), dtype=dout.dtype)
    np.put_along_axis(dxr, idx[..., None], dout[..., None], axis=-1)
    return (dxr.reshape(N, C, H // p, W_ // p, p, p)
                .transpose(0, 1, 2, 4, 3, 5)
                .reshape(N, C, H, W_))


# ── Check every gradient against PyTorch autograd ────────────────────
# Autograd is an independent implementation of the same derivatives, so agreement
# to ~1e-12 is strong evidence the hand-derived formulas are right.
xt = torch.tensor(x, requires_grad=True)
Wt = torch.tensor(Wf, requires_grad=True)
bt = torch.tensor(bf, requires_grad=True)

yt = F.conv2d(xt, Wt, bt, stride=1, padding=1)
g = rng.randn(*yt.shape)                       # arbitrary upstream gradient
yt.backward(torch.tensor(g))

_, cache = conv2d_fast(x, Wf, bf, stride=1, pad=1)
dx, dW, db = conv2d_backward(g, cache)

print('conv backward vs torch autograd:')
for name, mine, ref in [('dx', dx, xt.grad.numpy()),
                        ('dW', dW, Wt.grad.numpy()),
                        ('db', db, bt.grad.numpy())]:
    err = np.abs(mine - ref).max()
    print(f'  {name}: max |diff| = {err:.2e}   {"PASS" if err < 1e-10 else "FAIL"}')

# Max pooling, same treatment.
# Our maxpool assumes the window divides the spatial dims exactly, so trim 7x7 -> 6x6
# (torch would silently drop the remainder row/column instead).
xe = x[:, :, :6, :6]
xe_t = torch.tensor(xe, requires_grad=True)
yp = F.max_pool2d(xe_t, 2)
gp = rng.randn(*yp.shape)
yp.backward(torch.tensor(gp))

mine_out, pcache = maxpool_forward(xe, 2)
mine_dx = maxpool_backward(gp, pcache)
print('\nmaxpool vs torch:')
print(f'  forward : max |diff| = {np.abs(mine_out - yp.detach().numpy()).max():.2e}')
print(f'  backward: max |diff| = {np.abs(mine_dx - xe_t.grad.numpy()).max():.2e}')

## Finite-difference gradient check

The autograd comparison above already checks the backward pass against an independent
implementation. This adds the check that does not depend on PyTorch at all: nudge one weight by
$\pm\epsilon$, watch the loss move, and compare with the analytic gradient. Relative error below
$10^{-5}$ means backprop is right.

In [ ]:
# A complete tiny network -- conv -> ReLU -> maxpool -> linear -> softmax -- so the
# check exercises the conv gradients as they are actually used, with a real loss
# on top rather than an artificial upstream gradient.
def softmax(Z):
    e = np.exp(Z - Z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def tiny_net_loss(xg, yg, Wc, bc, Wl, bl, full=False):
    conv_out, conv_cache = conv2d_fast(xg, Wc, bc, stride=1, pad=1)
    relu_out = np.maximum(0, conv_out)
    pool_out, pool_cache = maxpool_forward(relu_out, 2)
    flat = pool_out.reshape(len(xg), -1)
    P = softmax(flat @ Wl + bl)
    loss = -np.mean(np.log(P[np.arange(len(yg)), yg] + 1e-12))
    if not full:
        return loss
    return loss, (conv_out, conv_cache, relu_out, pool_cache, flat, P)


rng = np.random.RandomState(1)
n_gc = 6
xg = rng.randn(n_gc, 1, 8, 8)
yg = rng.randint(0, 10, n_gc)
Wc = rng.randn(4, 1, 3, 3) * 0.5      # 4 conv filters
bc = rng.randn(4) * 0.1               # non-zero, so a bug in the bias path cannot hide
Wl = rng.randn(4 * 4 * 4, 10) * 0.3
bl = rng.randn(10) * 0.1

# ── Analytic gradients ───────────────────────────────────────────────
loss, (conv_out, conv_cache, relu_out, pool_cache, flat, P) = tiny_net_loss(
    xg, yg, Wc, bc, Wl, bl, full=True)

Y_oh = np.zeros_like(P); Y_oh[np.arange(n_gc), yg] = 1.0
dlogits = (P - Y_oh) / n_gc                       # softmax + cross-entropy, as always
dWl = flat.T @ dlogits
dbl = dlogits.sum(axis=0)
dflat = dlogits @ Wl.T
dpool = dflat.reshape(relu_out.shape[0], relu_out.shape[1],
                      relu_out.shape[2] // 2, relu_out.shape[3] // 2)
drelu = maxpool_backward(dpool, pool_cache)
dconv = drelu * (conv_out > 0)                    # ReLU derivative
dx_g, dWc, dbc = conv2d_backward(dconv, conv_cache)

# ── Numerical gradients ──────────────────────────────────────────────
eps = 1e-6
def numeric(param, ix):
    old = param[ix]
    param[ix] = old + eps; lp = tiny_net_loss(xg, yg, Wc, bc, Wl, bl)
    param[ix] = old - eps; lm = tiny_net_loss(xg, yg, Wc, bc, Wl, bl)
    param[ix] = old
    return (lp - lm) / (2 * eps)

print(f'loss = {loss:.6f}   (checking 20 random entries per tensor)\n')
for name, param, analytic in [('Wc (conv filters)', Wc, dWc),
                              ('bc (conv bias)   ', bc, dbc),
                              ('Wl (linear W)    ', Wl, dWl),
                              ('bl (linear bias) ', bl, dbl)]:
    idxs = list(zip(*[rng.randint(0, s, 20) for s in param.shape]))
    worst = 0.0
    for ix in idxs:
        num, ana = numeric(param, ix), analytic[ix]
        worst = max(worst, abs(num - ana) / (abs(num) + abs(ana) + 1e-12))
    print(f'{name}: max relative error = {worst:.2e}  '
          f'{"PASS" if worst < 1e-5 else "FAIL"}')

In [ ]:
# ── Train the from-scratch CNN on digits, NumPy only ─────────────────
# Architecture, identical to the PyTorch SmallCNN above:
#   (1,8,8) -> conv 8 filters 3x3 pad 1 -> ReLU -> maxpool 2x2 -> flatten(128) -> linear 10
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
Xd, yd = digits.data, digits.target


# Same stratified split as neural_networks.ipynb, in pure NumPy.
def stratified_split_numpy(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []
    for label_value in np.unique(y):
        class_indices = np.where(y == label_value)[0]
        shuffled = np.random.permutation(class_indices)
        n_test = int(len(shuffled) * test_size)
        test_idx.extend(shuffled[:n_test])
        train_idx.extend(shuffled[n_test:])
    # Shuffle ONCE and reuse the same order for X and y -- drawing a fresh
    # permutation for each would decouple images from their labels.
    train_idx = np.random.permutation(train_idx)
    test_idx = np.random.permutation(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


np.random.seed(42)
Xtr, Xte, ytr, yte = stratified_split_numpy(Xd, yd, 0.2, seed=42)
# Reshape to images and scale to [0, 1]; no per-pixel standardization (see the note
# in the PyTorch cell).
Xtr_i = (Xtr / 16.0).reshape(-1, 1, 8, 8)
Xte_i = (Xte / 16.0).reshape(-1, 1, 8, 8)

# Cheap guard: a mislabelled split trains to ~10% accuracy and looks like a broken
# gradient, so confirm the images still match their labels before training on them.
assert Xtr.shape[0] == ytr.shape[0] and Xte.shape[0] == yte.shape[0]
assert np.array_equal(np.sort(np.concatenate([ytr, yte])), np.sort(yd)), 'labels lost'
for probe in range(3):
    row = np.where((Xd == Xtr[probe]).all(axis=1))[0][0]
    assert yd[row] == ytr[probe], 'image/label mismatch in the split'
print('split check: images and labels still aligned')

# ── He initialization ────────────────────────────────────────────────
# fan_in for a conv filter is C_in * KH * KW -- the number of inputs feeding one
# output value. For 1 channel and 3x3 that is 9, not 64: the filter only ever
# sees 9 pixels at a time, however large the image is.
np.random.seed(0)
C_out, K = 8, 3
fan_in = 1 * K * K
Wc = np.random.randn(C_out, 1, K, K) * np.sqrt(2.0 / fan_in)
bc = np.zeros(C_out)
n_flat = C_out * 4 * 4                       # 8 channels of 4x4 after pooling
Wl = np.random.randn(n_flat, 10) * np.sqrt(2.0 / n_flat)
bl = np.zeros(10)

print(f'conv filters: {Wc.shape} -> {Wc.size} weights + {bc.size} biases')
print(f'linear      : {Wl.shape} -> {Wl.size} weights + {bl.size} biases')
print(f'total params: {Wc.size + bc.size + Wl.size + bl.size}\n')

# Plain mini-batch SGD, so the step size has to be larger than the PyTorch model
# needed -- Adam adapts its own per-parameter step, this does not.
lr, n_epochs, batch_size = 0.2, 60, 64
n = len(Xtr_i)
loss_curve_np = []

for epoch in range(n_epochs):
    perm = np.random.permutation(n)
    Xs, ys = Xtr_i[perm], ytr[perm]
    epoch_loss = 0.0

    for s in range(0, n, batch_size):
        xb, yb = Xs[s:s + batch_size], ys[s:s + batch_size]
        mb = len(yb)

        # ── Forward ──────────────────────────────────────────────
        conv_out, conv_cache = conv2d_fast(xb, Wc, bc, stride=1, pad=1)   # (mb,8,8,8)
        relu_out = np.maximum(0, conv_out)
        pool_out, pool_cache = maxpool_forward(relu_out, 2)               # (mb,8,4,4)
        flat = pool_out.reshape(mb, -1)                                   # (mb,128)
        P = softmax(flat @ Wl + bl)                                       # (mb,10)

        # ── Backward ─────────────────────────────────────────────
        Y_oh = np.zeros_like(P); Y_oh[np.arange(mb), yb] = 1.0
        dlogits = (P - Y_oh) / mb              # softmax + cross-entropy collapse
        dWl = flat.T @ dlogits
        dbl = dlogits.sum(axis=0)
        dflat = dlogits @ Wl.T
        dpool = dflat.reshape(pool_out.shape)  # un-flatten: pure reshape, no maths
        drelu = maxpool_backward(dpool, pool_cache)   # route to the argmax winners
        dconv = drelu * (conv_out > 0)                # ReLU: pass where input was > 0
        _, dWc, dbc = conv2d_backward(dconv, conv_cache)   # dx unused: nothing below

        # ── Update ───────────────────────────────────────────────
        Wc -= lr * dWc;  bc -= lr * dbc
        Wl -= lr * dWl;  bl -= lr * dbl

        epoch_loss += -np.sum(np.log(P[np.arange(mb), yb] + 1e-12))

    loss_curve_np.append(epoch_loss / n)
    if epoch % 10 == 0 or epoch == n_epochs - 1:
        print(f'Epoch {epoch:3d}: loss = {loss_curve_np[-1]:.4f}')


# ── Predict on the test set ──────────────────────────────────────────
def forward_eval(xb):
    c, _ = conv2d_fast(xb, Wc, bc, stride=1, pad=1)
    p, _ = maxpool_forward(np.maximum(0, c), 2)
    return softmax(p.reshape(len(xb), -1) @ Wl + bl)


P_test = forward_eval(Xte_i)
y_hat = P_test.argmax(axis=1)

accuracy = np.mean(y_hat == yte)
test_loss = -np.mean(np.log(P_test[np.arange(len(yte)), yte] + 1e-12))
print(f'\nTest accuracy:      {accuracy:.4f}')
print(f'Test cross-entropy: {test_loss:.4f}')

# ── Manual metrics, same layout as neural_networks.ipynb ─────────────
K_cls = 10
print(f'\n{"Class":>5}  {"Precision":>9}  {"Recall":>6}  {"F1":>6}  {"Support":>7}')
print('-' * 44)
for k in range(K_cls):
    tp = np.sum((y_hat == k) & (yte == k))
    fp = np.sum((y_hat == k) & (yte != k))
    fn = np.sum((y_hat != k) & (yte == k))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    print(f'{k:5d}  {prec:9.4f}  {rec:6.4f}  {f1:6.4f}  {int(np.sum(yte == k)):7d}')

cm = np.zeros((K_cls, K_cls), dtype=int)
for t, p_ in zip(yte, y_hat):
    cm[t, p_] += 1
print(f'\nConfusion matrix (rows = actual, cols = predicted):\n{cm}')

plt.figure(figsize=(6, 4))
plt.plot(loss_curve_np, color='indigo', lw=2)
plt.title('From-scratch CNN: training loss')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy')
plt.show()

## What this notebook shows, and what it does not

### The implementation is verified, not asserted

| check | result |
| --- | --- |
| naive loops vs PyTorch `F.conv2d` (4 stride/pad combinations) | ~$3\times10^{-15}$ |
| `im2col` vs naive vs PyTorch | ~$3\times10^{-15}$ |
| `dx`, `dW`, `db` vs PyTorch autograd | $\le 7\times10^{-15}$ |
| max pool forward and backward vs PyTorch | exactly $0$ |
| all four tensors vs finite differences | $\le 6\times10^{-6}$ (threshold $10^{-5}$) |

And `im2col` is ~130× faster than the naive loops on a realistic batch, computing exactly the same
numbers — PyTorch is ~1000× faster still.

### On 8×8 digits, convolution does not win on accuracy

The PyTorch CNN and the MLP from `neural_networks.ipynb` both land on **0.9778**. Identical. The
from-scratch NumPy CNN reaches **0.9634**, the same as the from-scratch MLP — the residual gap to
the framework version is plain SGD versus Adam, not a difference in the convolution.

That is the honest result, and the reasons are worth naming:

- The images are 8×8 and the digits are already centred, so there is very little translation for the
  architecture to be robust to.
- At this size a 3×3 window already covers a large fraction of the image, so "local" and "global"
  barely differ.
- 1,437 training images of an easy 10-class problem is a regime where most reasonable models score
  in the high nineties.

### What convolution does buy, measurably, here

**Parameters.** 1,370 against the MLP's 9,610 — about 7× fewer for identical accuracy. The
convolution itself is 80 of them: 8 filters × 9 weights, plus 8 biases.

**Robustness to shift.** Shifting the test digits one pixel right costs the MLP **−0.58** accuracy
but the CNN only **−0.37**, and the global-average-pool CNN just **−0.06**. This is the property
that starts mattering the moment objects are not perfectly centred.

**Independence from image size.** Those 3×3 filters would be unchanged on 256×256 inputs. A dense
first layer would need ~8 million weights for the same 128 hidden units.

### The result that complicates the story

The pixel-shuffle experiment did **not** come out the way the motivation section predicts. Scrambling
the pixels barely dented the CNN with a dense head (−0.009) — because its 1,290-weight fully-connected
layer is strong enough to classify from scrambled features even when the convolution below it has
been rendered meaningless. Only when the head is replaced by global average pooling, which discards
position and so cannot compensate, does the damage appear clearly (−0.075).

The lesson generalizes beyond this notebook: **an architectural prior only helps to the extent the
rest of the network relies on it.** Bolting a large dense layer on top of a conv stack can quietly
undo much of what the convolution was there to provide. It is also a reminder to average over seeds —
a single run of this experiment showed the CNN *improving* under shuffling, which is pure noise on a
360-image test set.

### Where convolution actually pays off

Larger images, objects that move, and depth. Stacked conv layers build a hierarchy — early filters
find edges, later ones combine edges into strokes, then strokes into shapes — with each layer seeing
a wider region of the original image than the last. That is the LeNet → AlexNet → VGG → ResNet
progression in `models.md`, and it is the next rung after this notebook.

### Deliberately not covered

Deep conv stacks, dilation, 1×1 convolutions, strided convolution as a pooling replacement,
transposed convolution, and batch norm on conv layers (which normalizes per *channel* rather than per
feature — see `makemore_3_bn.ipynb` for the dense case, and the theory section of
`neural_networks.ipynb`). The goal here was one convolution, understood completely, rather than a
survey.